# 🚀 Partition Pruning in Spark (Deep Dive for Interviews)

---

# 1️⃣ What is Partition Pruning?

## 📌 Definition

> Partition pruning is an optimization technique where Spark reads only the required partitions instead of scanning the entire dataset.

---

## 🔹 Key Idea

Instead of reading:

```
All data → Filter later
```

Spark does:

```
Filter first → Read only required partitions
```

---

# 2️⃣ Example (Very Important)

---

## 🔹 Data Stored as Partitioned Files

```
/data/
   ├── country=IN/
   ├── country=US/
   ├── country=UK/
```

---

## 🔹 Query

```python
df.filter("country = 'IN'")
```

---

## 🔥 Without Partition Pruning

- Reads all folders (IN, US, UK)  
- Filters later ❌  

---

## 🔥 With Partition Pruning

- Reads only:

```
/data/country=IN/
```

👉 Huge performance gain ✅

---

# 3️⃣ How Spark Does Partition Pruning

---

## 🔹 Step-by-Step

1. Detect partition column  
2. Analyze filter condition  
3. Identify required partitions  
4. Skip irrelevant files  

---

## 🔹 Requirement

👉 Column must be **partition column**

---

# 4️⃣ Hands-On Example

---

## 🔹 Writing Partitioned Data

```python
df.write.partitionBy("country").parquet("/data")
```

---

## 🔹 Reading with Filter

```python
df = spark.read.parquet("/data")

df.filter("country = 'IN'").show()
```

---

## 🔹 Check Execution Plan

```python
df.filter("country = 'IN'").explain(True)
```

Look for:

```
PushedFilters / PartitionFilters
```

---

# 5️⃣ Partition Pruning in SQL

---

```sql
SELECT * FROM table
WHERE country = 'IN';
```

---

👉 Only required partition scanned

---

# 6️⃣ Partition Pruning in Delta Lake

---

## 🔹 Example

```python
df.write.format("delta") \
  .partitionBy("country") \
  .save("/delta/table")
```

---

## 🔹 Query

```python
spark.read.format("delta") \
  .load("/delta/table") \
  .filter("country = 'IN'")
```

---

👉 Delta uses:

- Partition pruning  
- Data skipping (extra optimization)

---

# 7️⃣ Partition Pruning vs Predicate Pushdown

---

| Feature | Partition Pruning | Predicate Pushdown |
|----------|------------------|---------------------|
| Level | File/folder | Data inside file |
| Works on | Partition column | Any column |
| Benefit | Skip files | Reduce row scan |

---

## 🔹 Example

```python
df.filter("country = 'IN' AND age > 30")
```

- country → Partition pruning  
- age → Predicate pushdown  

---

# 8️⃣ When Partition Pruning DOES NOT Work

---

## ❌ Case 1: No Partition Column

```python
df.filter("age > 30")
```

---

## ❌ Case 2: Transformation on Column

```python
df.filter("UPPER(country) = 'IN'")
```

👉 Breaks pruning

---

## ❌ Case 3: Wrong Data Type

```python
df.filter("country = 1")
```

---

# 9️⃣ Best Practices

---

- Always partition on frequently filtered columns  
- Avoid over-partitioning  
- Use low to medium cardinality columns  

---

## 🔹 Good Partition Column

- country  
- date  
- region  

---

## ❌ Bad Partition Column

- user_id (too many partitions)  

---

# 🔟 Real-World Example

---

## Scenario

- 1 TB dataset partitioned by date  
- Query:

```sql
SELECT * FROM sales WHERE date = '2025-01-01'
```

---

## 🔹 Without Pruning

- Reads 1 TB ❌  

---

## 🔹 With Pruning

- Reads only one partition (~10 GB) ✅  

---

# 1️⃣1️⃣ Interview Questions

---

## ❓ What is partition pruning?

👉 Skipping unnecessary partitions during read.

---

## ❓ How does Spark achieve it?

👉 By analyzing partition columns and filter conditions.

---

## ❓ Difference between partition pruning and predicate pushdown?

👉 Pruning → file level  
👉 Pushdown → row level  

---

## ❓ Why is partition pruning important?

👉 Reduces I/O and improves performance.

---

## ❓ When does pruning fail?

👉 When filters are not on partition columns or transformed.

---

# 🎯 Interview Answer (Best)

Partition pruning is an optimization technique in Spark where only relevant partitions are read based on filter conditions applied on partition columns, significantly reducing I/O and improving performance.

---

# 🚀 Final Summary

```
Partition Column → Filter applied
        ↓
Spark skips unnecessary partitions
        ↓
Less data read → Faster query
```

---

# 🔥 Golden Rule

👉 Filter on partition column = Massive performance gain 🚀  

# 🚀 Column Pruning in Spark (Deep Dive for Interviews)

---

# 1️⃣ What is Column Pruning?

## 📌 Definition

> Column pruning is an optimization technique where Spark reads only the required columns instead of scanning the entire dataset.

---

## 🔹 Key Idea

Instead of:

```
Read all columns → Use few
```

Spark does:

```
Read only required columns
```

---

# 2️⃣ Example (Very Important)

---

## 🔹 Dataset

| id | name | age | salary |
|----|------|-----|--------|

---

## 🔹 Query

```python
df.select("id", "name")
```

---

## 🔥 Without Column Pruning

- Reads all columns ❌  
- Wastes memory and I/O  

---

## 🔥 With Column Pruning

- Reads only:

```
id, name
```

👉 Faster execution ✅

---

# 3️⃣ How Column Pruning Works

---

## 🔹 Step-by-Step

1. Spark analyzes query  
2. Identifies required columns  
3. Pushes projection to data source  
4. Reads only selected columns  

---

## 🔹 Requirement

👉 Works best with **columnar formats**

- Parquet ✅  
- ORC ✅  
- Delta ✅  

---

# 4️⃣ Hands-On Example

---

## 🔹 Read Data

```python
df = spark.read.parquet("/data")
```

---

## 🔹 Apply Column Selection

```python
df.select("id", "name").show()
```

---

## 🔹 Check Execution Plan

```python
df.select("id", "name").explain(True)
```

Look for:

```
Project [id, name]
```

---

# 5️⃣ Column Pruning in SQL

---

```sql
SELECT id, name FROM table;
```

---

👉 Only required columns are read

---

# 6️⃣ Column Pruning in Delta Lake

---

```python
spark.read.format("delta") \
  .load("/delta/table") \
  .select("id", "name")
```

---

👉 Delta uses:

- Column pruning  
- Data skipping  

---

# 7️⃣ Column Pruning vs Partition Pruning

---

| Feature | Column Pruning | Partition Pruning |
|----------|----------------|--------------------|
| Level | Column | Partition |
| Reduces | Data read | Files scanned |
| Works on | Any column | Partition column only |

---

## 🔹 Example

```python
df.filter("country = 'IN'").select("id", "name")
```

- country → Partition pruning  
- id, name → Column pruning  

---

# 8️⃣ When Column Pruning Fails

---

## ❌ Case 1: Using SELECT *

```python
df.select("*")
```

👉 Reads all columns  

---

## ❌ Case 2: Complex Transformations

```python
df.selectExpr("concat(name, '_test')")
```

👉 May reduce optimization  

---

## ❌ Case 3: Non-columnar formats

- CSV ❌  
- JSON ❌  

---

# 9️⃣ Best Practices

---

- Avoid `SELECT *`  
- Select only required columns  
- Use columnar formats (Parquet/Delta)  
- Combine with partition pruning  

---

# 🔟 Real-World Example

---

## Scenario

Dataset:

- 100 columns  
- 1 TB size  

Query:

```sql
SELECT id, name FROM table;
```

---

## 🔹 Without Column Pruning

- Reads 1 TB ❌  

---

## 🔹 With Column Pruning

- Reads only required columns (~50 GB) ✅  

---

# 1️⃣1️⃣ Interview Questions

---

## ❓ What is column pruning?

👉 Reading only required columns instead of full dataset.

---

## ❓ Why is it important?

👉 Reduces I/O, memory usage, and improves performance.

---

## ❓ When does it work best?

👉 Columnar formats like Parquet/Delta.

---

## ❓ Difference from partition pruning?

👉 Column pruning → columns  
👉 Partition pruning → partitions  

---

## ❓ Why avoid SELECT *?

👉 Disables column pruning and increases cost.

---

# 🎯 Interview Answer (Best)

Column pruning is an optimization technique in Spark where only the required columns are read from the data source, reducing I/O and improving performance, especially in columnar formats like Parquet and Delta.

---

# 🚀 Final Summary

```
Select only needed columns
        ↓
Spark reads fewer columns
        ↓
Less I/O → Faster queries
```

---

# 🔥 Golden Rule

👉 Avoid SELECT * → Always select required columns only 🚀  